# 🎙️ Fine-tuning Distil-Whisper on TORGO Dysarthric Speech (Kaggle Free Tier)

**Model:** `distil-whisper/distil-large-v3`  
**Dataset:** `abnerh/TORGO-database`  
**Target:** Adapt Whisper for dysarthric (motor-impaired) speech recognition  
**Hardware:** Kaggle Free Tier (T4 GPU, 16 GB VRAM, ~20 GB disk, 9-hour session limit)

---

## 📋 Notebook Overview

| Section | What it does |
|---------|-------------|
| 0. Setup | Install deps, HF auth via Kaggle Secrets |
| 1. Config | All hyperparameters in one place |
| 2. Dataset | Load TORGO, split, preprocess |
| 3. Model | Load distil-whisper with correct dtype settings |
| 4. Pilot run | Train on 500 samples — fast sanity check |
| 5. Full run | Train on entire dataset |
| 6. Evaluation | WER/CER on test split |
| 7. Inference | Interactive demo cell |
| 8. Hub upload | Push model to Hugging Face |
| 9. Troubleshooting | Common failures + fixes |

---

## ⚠️ Kaggle-specific setup (read before running)

### GPU
Settings → Accelerator → **GPU T4 x1**

### HF Token Secret
1. Go to **Add-ons → Secrets** (top menu) or the key icon in the sidebar  
2. Add a secret named **`HF_TOKEN`** with your token from https://huggingface.co/settings/tokens  
3. Make sure **"Notebook access"** is toggled ON for the secret  
4. Requires write permission to push the final model to the Hub

### Kaggle vs Colab differences (all handled automatically)
- ✅ No Google Drive — checkpoints go to `/kaggle/working/` (persisted as output)
- ✅ Secrets via `kaggle_secrets` instead of `google.colab.userdata`
- ✅ No `google.colab.files` — audio upload uses a local path instead
- ✅ `IPython.display.Audio` works the same
- ✅ Session limit is ~9 hrs — pilot + full training fits if you skip re-runs

### Known issues fixed in this notebook
- ✅ `processing_class` used instead of deprecated `tokenizer` arg
- ✅ Encoder kept in FP32 to avoid FP16 gradient unscaling crash
- ✅ Gradient checkpointing disabled when encoder is unfrozen
- ✅ WER metric computed correctly (normalised, not empty-string collapsed)
- ✅ Inference pipeline configured without `forced_decoder_ids` conflict
- ✅ Dataset shuffled before split (TORGO is class-ordered)
- ✅ Auto-resume from latest `/kaggle/working/` checkpoint


## 0. Setup & Authentication

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 0.1 — Install all required packages
# Kaggle kernels ship with many ML packages pre-installed, but we pin
# versions that are known to work together with distil-whisper as of mid-2025.
# ─────────────────────────────────────────────────────────────────────────────

!pip install -q \
    transformers==4.44.2 \
    datasets==2.21.0 \
    accelerate==0.34.2 \
    evaluate==0.4.3 \
    jiwer==3.0.4 \
    soundfile==0.12.1 \
    librosa==0.10.2 \
    huggingface_hub==0.26.0 \
    tensorboard==2.17.1

print("✅ Packages installed")

✅ Packages installed


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 0.2 — Set up working directories
#
# Kaggle differences from Colab:
#   • No Google Drive — use /kaggle/working/ instead.
#   • /kaggle/working/ is PERSISTENT across sessions as a Kaggle Output.
#     Checkpoints saved here survive the session and can be downloaded or
#     reused in a new session by adding the previous run as a dataset input.
#   • /kaggle/input/ contains read-only datasets you have attached.
#   • Disk quota: ~20 GB for /kaggle/working/ on the free tier.
# ─────────────────────────────────────────────────────────────────────────────

import os

# Base directory for all training artefacts
WORKING_BASE = "/kaggle/working/whisper_torgo"
os.makedirs(WORKING_BASE, exist_ok=True)
print(f"✅ Working directory ready — artefacts → {WORKING_BASE}")

✅ Working directory ready — artefacts → /kaggle/working/whisper_torgo


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 0.3 — Hugging Face login
#
# Kaggle Secrets replaces google.colab.userdata.
# To add your token: Add-ons → Secrets → + Add a new secret
#   Name : HF_TOKEN
#   Value: hf_...  (your write-access token)
#   Toggle "Notebook access" ON for this notebook.
# ─────────────────────────────────────────────────────────────────────────────

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    login(token=hf_token, add_to_git_credential=True)
    print("✅ Logged in via Kaggle Secret")
except Exception as e:
    print(f"⚠️  Could not read HF_TOKEN from Kaggle Secrets: {e}")
    print("   Add it via Add-ons → Secrets, or uncomment the line below:")
    # login(token="hf_YOUR_TOKEN_HERE")
    raise

Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
✅ Logged in via Kaggle Secret


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 0.4 — Verify GPU
#
# Make sure the accelerator is enabled:
#   Settings (right panel) → Accelerator → GPU T4 x1
# ─────────────────────────────────────────────────────────────────────────────

import torch

assert torch.cuda.is_available(), (
    "❌ No GPU found! Enable it via Settings → Accelerator → GPU T4 x1"
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU: {gpu_name}")
print(f"   VRAM: {vram_gb:.1f} GB")
# T4 has 15-16 GB VRAM on Kaggle — our config is tuned to stay under ~13 GB

✅ GPU: Tesla T4
   VRAM: 15.6 GB


## 1. Configuration

**All hyperparameters live here.** Change values in this cell; everything downstream picks them up automatically.

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1.1 — Master config
# ─────────────────────────────────────────────────────────────────────────────

from dataclasses import dataclass
from typing import Optional

@dataclass
class Config:
    # ── Model ─────────────────────────────────────────────────────────────────
    model_name: str = "openai/whisper-tiny"
    # Your HF username/repo-name for the final push
    hub_model_id: str = "ruch9265/distil-whisper-torgo"

    # ── Dataset ───────────────────────────────────────────────────────────────
    dataset_name: str = "abnerh/TORGO-database"
    # TORGO only has a train split — we create test/val ourselves
    test_size: float = 0.10   # 10% held-out test
    val_size:  float = 0.10   # 10% validation (from remaining 90%)
    dataset_seed: int = 42

    # ── Audio ────────────────────────────────────────────────────────────────
    sampling_rate: int = 16_000   # Whisper expects 16 kHz
    max_duration_s: float = 30.0
    min_duration_s: float = 0.1

    # ── Training — T4-safe defaults ───────────────────────────────────────────
    # KAGGLE: paths point to /kaggle/working/ instead of Google Drive
    output_dir:  str = "/kaggle/working/whisper_torgo/checkpoints"
    logging_dir: str = "/kaggle/working/whisper_torgo/logs"

    # Effective batch size = per_device_train_batch_size × gradient_accumulation_steps
    # = 8 × 4 = 32  (sweet spot for T4 VRAM)
    per_device_train_batch_size: int = 8
    per_device_eval_batch_size:  int = 8
    gradient_accumulation_steps: int = 2

    max_steps:     int = 2000
    warmup_steps:  int = 100
    save_steps:    int = 200
    eval_steps:    int = 200
    logging_steps: int = 25

    learning_rate:       float = 5e-6
    weight_decay:        float = 0.01
    lr_scheduler_type:   str   = "linear"

    # Mixed precision: fp16 on T4 GPU
    fp16: bool = True

    # IMPORTANT: gradient_checkpointing is OFF.
    # When the full encoder is trainable + fp16 is on, gradient checkpointing
    # causes: "ValueError: Attempting to unscale FP16 gradients".
    gradient_checkpointing: bool = False

    # Freeze encoder? NO — acoustic adaptation for dysarthric speech
    # requires encoder fine-tuning, and frozen encoder + fp16 = crash.
    freeze_encoder: bool = False

    save_total_limit:       int  = 3
    load_best_model_at_end: bool = True
    metric_for_best_model:  str  = "wer"
    greater_is_better:      bool = False  # lower WER = better

    # ── Pilot run ─────────────────────────────────────────────────────────────
    pilot_samples:    int = 600
    pilot_max_steps:  int = 100
    # KAGGLE: pilot checkpoints also go to /kaggle/working/
    pilot_output_dir: str = "/kaggle/working/whisper_torgo/pilot_checkpoints"

    # ── Language ─────────────────────────────────────────────────────────────
    language: str = "english"
    task:     str = "transcribe"


cfg = Config()

print("=" * 60)
print("CONFIG SUMMARY")
print("=" * 60)
print(f"  Model        : {cfg.model_name}")
print(f"  Dataset      : {cfg.dataset_name}")
print(f"  Hub target   : {cfg.hub_model_id}")
print(f"  Effective BS : {cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps}")
print(f"  Max steps    : {cfg.max_steps}")
print(f"  LR           : {cfg.learning_rate}")
print(f"  FP16         : {cfg.fp16}")
print(f"  Grad ckpt    : {cfg.gradient_checkpointing}")
print(f"  Freeze enc   : {cfg.freeze_encoder}")
print(f"  Output dir   : {cfg.output_dir}")
print("=" * 60)

CONFIG SUMMARY
  Model        : openai/whisper-tiny
  Dataset      : abnerh/TORGO-database
  Hub target   : ruch9265/distil-whisper-torgo
  Effective BS : 16
  Max steps    : 2000
  LR           : 5e-06
  FP16         : True
  Grad ckpt    : False
  Freeze enc   : False
  Output dir   : /kaggle/working/whisper_torgo/checkpoints


## 2. Dataset Loading & Preprocessing

TORGO only ships a `train` split. We shuffle first (it's class-ordered) then carve out our own test + validation sets.

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2.1 — Load raw dataset
# datasets caches to ~/.cache/huggingface/datasets by default.
# On Kaggle the cache lives on the ephemeral /root disk (~100 GB+),
# so no redirect needed. Audio is decoded lazily on first access.
# ─────────────────────────────────────────────────────────────────────────────

from datasets import load_dataset, Audio

print("Loading TORGO dataset (this streams audio metadata, ~1-2 min)...")

raw = load_dataset(
    cfg.dataset_name,
    split="train",           # only split available
    trust_remote_code=True,  # required for custom loading scripts
)

print(f"Loaded {len(raw):,} samples")
print(f"Features: {raw.features}")
print("\nSample row:")
print(raw[0])

Loading TORGO dataset (this streams audio metadata, ~1-2 min)...
Loaded 16,552 samples
Features: {'audio': Audio(sampling_rate=None, mono=True, decode=True, id=None), 'transcription': Value(dtype='string', id=None), 'speech_status': Value(dtype='string', id=None), 'gender': Value(dtype='string', id=None), 'duration': Value(dtype='float64', id=None)}

Sample row:
{'audio': {'path': 'FC01_1_arrayMic_0066.wav', 'array': array([ 0.00125122,  0.00387573,  0.00115967, ...,  0.00149536,
       -0.00326538,  0.00027466]), 'sampling_rate': 16000}, 'transcription': 'alpha', 'speech_status': 'healthy', 'gender': 'female', 'duration': 3.3}


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2.2 — Inspect class distribution
# ─────────────────────────────────────────────────────────────────────────────

from collections import Counter
import statistics

status_counts = Counter(raw["speech_status"])
gender_counts = Counter(raw["gender"])
durations     = raw["duration"]

print("Speech status distribution:")
for k, v in status_counts.items():
    print(f"  {k:12s}: {v:6,d} ({100*v/len(raw):.1f}%)")

print("\nGender distribution:")
for k, v in gender_counts.items():
    print(f"  {k:12s}: {v:6,d} ({100*v/len(raw):.1f}%)")

print(f"\nDuration stats (seconds):")
print(f"  min    = {min(durations):.2f}")
print(f"  max    = {max(durations):.2f}")
print(f"  mean   = {statistics.mean(durations):.2f}")
print(f"  median = {statistics.median(durations):.2f}")

Speech status distribution:
  healthy     : 10,978 (66.3%)
  dysarthria  :  5,574 (33.7%)

Gender distribution:
  female      :  6,407 (38.7%)
  male        : 10,145 (61.3%)

Duration stats (seconds):
  min    = 0.15
  max    = 193.83
  mean   = 2.97
  median = 2.53


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2.3 — Filter outliers & shuffle
#
# CRITICAL: the original dataset is class-ordered (healthy first, then
# dysarthric). We MUST shuffle before splitting or our test set will be
# entirely dysarthric and our train set entirely healthy — a disaster.
# ─────────────────────────────────────────────────────────────────────────────

n_before = len(raw)

filtered = raw.filter(
    lambda x: cfg.min_duration_s <= x["duration"] <= cfg.max_duration_s,
    desc="Filtering duration outliers",
)

print(f"Removed {n_before - len(filtered):,} samples outside "
      f"[{cfg.min_duration_s}, {cfg.max_duration_s}] s")
print(f"Remaining: {len(filtered):,}")

shuffled = filtered.shuffle(seed=cfg.dataset_seed)
print(f"Shuffled dataset (seed={cfg.dataset_seed})")

Removed 2 samples outside [0.1, 30.0] s
Remaining: 16,550
Shuffled dataset (seed=42)


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2.4 — Create train / validation / test splits
# ─────────────────────────────────────────────────────────────────────────────

# Step 1: test split
tmp = shuffled.train_test_split(
    test_size=cfg.test_size,
    seed=cfg.dataset_seed,
)

# Step 2: validation split from training portion
train_val = tmp["train"].train_test_split(
    test_size=cfg.val_size / (1.0 - cfg.test_size),
    seed=cfg.dataset_seed,
)

ds_train = train_val["train"]
ds_val   = train_val["test"]
ds_test  = tmp["test"]

print(f"Split sizes:")
print(f"  train : {len(ds_train):6,d}")
print(f"  val   : {len(ds_val):6,d}")
print(f"  test  : {len(ds_test):6,d}")

for name, ds in [("train", ds_train), ("val", ds_val), ("test", ds_test)]:
    c = Counter(ds["speech_status"])
    total = sum(c.values())
    print(f"  {name}: healthy={c['healthy']:,} ({100*c['healthy']/total:.0f}%) "
          f"dysarthric={c['dysarthria']:,} ({100*c['dysarthria']/total:.0f}%)")

Split sizes:
  train : 13,239
  val   :  1,656
  test  :  1,655
  train: healthy=8,746 (66%) dysarthric=4,493 (34%)
  val: healthy=1,112 (67%) dysarthric=544 (33%)
  test: healthy=1,120 (68%) dysarthric=535 (32%)


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2.5 — Load feature extractor + tokenizer + processor
# ─────────────────────────────────────────────────────────────────────────────

from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    cfg.model_name,
    language=cfg.language,
    task=cfg.task,
)

feature_extractor = processor.feature_extractor
tokenizer         = processor.tokenizer

print(f"✅ Processor loaded")
print(f"   Sampling rate : {feature_extractor.sampling_rate}")
print(f"   Language token: {tokenizer.language}")
print(f"   Task token    : {tokenizer.task}")

✅ Processor loaded
   Sampling rate : 16000
   Language token: english
   Task token    : transcribe


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2.6 — Resample audio to 16 kHz
# ─────────────────────────────────────────────────────────────────────────────

ds_train = ds_train.cast_column("audio", Audio(sampling_rate=cfg.sampling_rate))
ds_val   = ds_val.cast_column("audio",   Audio(sampling_rate=cfg.sampling_rate))
ds_test  = ds_test.cast_column("audio",  Audio(sampling_rate=cfg.sampling_rate))

print(f"✅ Audio columns cast to {cfg.sampling_rate} Hz")

✅ Audio columns cast to 16000 Hz


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2.7 — Preprocessing function
#
# WHY return_attention_mask=False:
# Whisper's feature extractor always pads to exactly 3000 frames (30 s).
# The attention mask would be all-ones for every sample, so it's redundant.
# ─────────────────────────────────────────────────────────────────────────────

def preprocess_batch(batch):
    audio_arrays = [sample["array"] for sample in batch["audio"]]

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=cfg.sampling_rate,
        return_tensors="pt",
        return_attention_mask=False,
    )
    batch["input_features"] = inputs.input_features

    labels = tokenizer(
        batch["transcription"],
        padding=False,
    ).input_ids
    batch["labels"] = labels

    return batch


print("Preprocessing train split...")
ds_train_proc = ds_train.map(
    preprocess_batch,
    batched=True,
    batch_size=32,
    remove_columns=ds_train.column_names,
    num_proc=2,
    desc="Train preprocessing",
)

print("Preprocessing val split...")
ds_val_proc = ds_val.map(
    preprocess_batch,
    batched=True,
    batch_size=32,
    remove_columns=ds_val.column_names,
    num_proc=2,
    desc="Val preprocessing",
)

print("Preprocessing test split...")
ds_test_proc = ds_test.map(
    preprocess_batch,
    batched=True,
    batch_size=32,
    remove_columns=ds_test.column_names,
    num_proc=2,
    desc="Test preprocessing",
)

print(f"✅ Preprocessing complete")
print(f"   train: {len(ds_train_proc):,} | val: {len(ds_val_proc):,} | test: {len(ds_test_proc):,}")

Preprocessing train split...
Preprocessing val split...
Preprocessing test split...
✅ Preprocessing complete
   train: 13,239 | val: 1,656 | test: 1,655


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2.8 — Data collator
# ─────────────────────────────────────────────────────────────────────────────

from dataclasses import dataclass as dc
from typing import Any, Dict, List
import torch


@dc
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


from transformers import WhisperConfig
_config = WhisperConfig.from_pretrained(cfg.model_name)

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=_config.decoder_start_token_id,
)

print(f"✅ Data collator ready")
print(f"   decoder_start_token_id = {_config.decoder_start_token_id}")

✅ Data collator ready
   decoder_start_token_id = 50258


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2.8b — Cache preprocessed dataset to HF Hub (run once, skip next time)
#
# Preprocessing (Cell 2.7) takes ~20–30 min. This cell saves the processed
# Arrow dataset to HF Hub so future sessions can skip preprocessing entirely.
#
# On first run  : pushes ds_{train,val,test}_proc to Hub as a dataset repo.
# On resume     : loads from Hub, skipping the entire Section 2 preprocessing.
#
# Set SKIP_DATASET_CACHE = True to disable this behaviour.
# ─────────────────────────────────────────────────────────────────────────────

from datasets import load_from_disk, DatasetDict
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

SKIP_DATASET_CACHE = False   # ← set True to always re-preprocess

# Use a separate dataset repo: <username>/torgo-whisper-processed
DATASET_HUB_ID = cfg.hub_model_id.split("/")[0] + "/torgo-whisper-processed"
DATASET_LOCAL_CACHE = os.path.join(WORKING_BASE, "processed_dataset")

def try_load_cached_dataset(hub_id, local_cache):
    """Returns (ds_train_proc, ds_val_proc, ds_test_proc) or None if not cached."""
    # Try local disk first (same session, no re-download)
    if os.path.isdir(os.path.join(local_cache, "train")):
        print(f"✅ Loading preprocessed dataset from local cache: {local_cache}")
        dd = load_from_disk(local_cache)
        return dd["train"], dd["val"], dd["test"]
    # Try HF Hub
    try:
        from huggingface_hub import snapshot_download
        print(f"Downloading preprocessed dataset from HF Hub ({hub_id})...")
        snapshot_download(
            repo_id=hub_id,
            repo_type="dataset",
            local_dir=local_cache,
            ignore_patterns=["*.md"],
        )
        dd = load_from_disk(local_cache)
        print(f"✅ Loaded from HF Hub — skipping preprocessing!")
        return dd["train"], dd["val"], dd["test"]
    except Exception as e:
        print(f"   No cached dataset on Hub ({e}). Will preprocess fresh.")
        return None


if not SKIP_DATASET_CACHE:
    cached = try_load_cached_dataset(DATASET_HUB_ID, DATASET_LOCAL_CACHE)
    if cached:
        ds_train_proc, ds_val_proc, ds_test_proc = cached
        print(f"   train: {len(ds_train_proc):,} | val: {len(ds_val_proc):,} | test: {len(ds_test_proc):,}")
        print("   ⏩ Preprocessing cells below will be skipped automatically.")
        _DATASET_FROM_CACHE = True
    else:
        _DATASET_FROM_CACHE = False
else:
    _DATASET_FROM_CACHE = False
    print("SKIP_DATASET_CACHE=True — will preprocess from scratch.")


✅ Loading preprocessed dataset from local cache: /kaggle/working/whisper_torgo/processed_dataset


Loading dataset from disk:   0%|          | 0/26 [00:00<?, ?it/s]

   train: 13,239 | val: 1,656 | test: 1,655
   ⏩ Preprocessing cells below will be skipped automatically.


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2.9 — Push preprocessed dataset to HF Hub (run after Cell 2.7)
#
# Only run this ONCE after preprocessing completes successfully.
# Skip if you already see torgo-whisper-processed in your HF profile.
# ─────────────────────────────────────────────────────────────────────────────

if not _DATASET_FROM_CACHE:
    from datasets import DatasetDict
    from huggingface_hub import create_repo

    create_repo(DATASET_HUB_ID, repo_type="dataset", exist_ok=True)

    dd = DatasetDict({
        "train": ds_train_proc,
        "val":   ds_val_proc,
        "test":  ds_test_proc,
    })

    # Save locally first (Arrow format)
    os.makedirs(DATASET_LOCAL_CACHE, exist_ok=True)
    dd.save_to_disk(DATASET_LOCAL_CACHE)
    print(f"✅ Saved locally to {DATASET_LOCAL_CACHE}")

    # Push to HF Hub
    print(f"Pushing to HF Hub dataset repo: {DATASET_HUB_ID} ...")
    dd.push_to_hub(DATASET_HUB_ID, private=True)
    print(f"✅ Preprocessed dataset pushed to https://huggingface.co/datasets/{DATASET_HUB_ID}")
    print("   Future sessions will load from here — skipping preprocessing entirely.")
else:
    print("Dataset was loaded from cache — nothing to push.")


Dataset was loaded from cache — nothing to push.


## 3. Model Setup

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3.1 — Load model
#
# KEY DTYPE DECISION:
# Load in FP32 (default). Trainer + fp16=True handles AMP correctly.
# DO NOT pass torch_dtype=torch.float16 — it causes a GradScaler crash.
# DO NOT freeze the encoder — dysarthric adaptation requires it to update.
# ─────────────────────────────────────────────────────────────────────────────

# KEY DTYPE DECISION:
# Load in FP32 (default). Trainer + fp16=True handles AMP correctly.
# DO NOT pass torch_dtype=torch.float16 — it causes a GradScaler crash.
# DO NOT freeze the encoder — dysarthric adaptation requires it to update.
#
# DECODER IS FROZEN — TORGO transcriptions are plain English sentences so
# the decoder needs no adaptation. Freezing it halves trainable params,
# reduces VRAM, speeds up backprop, and prevents overfitting on 3500 samples.

from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(cfg.model_name)

model.generation_config.language         = cfg.language
model.generation_config.task             = cfg.task
model.generation_config.forced_decoder_ids = None  # avoids silent truncation
model.generation_config.suppress_tokens  = []      # avoids empty transcriptions
for param in model.model.decoder.parameters():
    param.requires_grad = False

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model loaded: {cfg.model_name}")
print(f"   Total params    : {total_params/1e6:.1f} M")
print(f"   Trainable params: {trainable_params/1e6:.1f} M")
print(f"   Frozen params   : {(total_params - trainable_params)/1e6:.1f} M")

✅ Model loaded: openai/whisper-tiny
   Total params    : 37.8 M
   Trainable params: 7.6 M
   Frozen params   : 30.1 M


In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3.2 — Metrics (WER + CER)
# ─────────────────────────────────────────────────────────────────────────────

import evaluate
import numpy as np

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str  = [p.lower().strip() for p in pred_str]
    label_str = [l.lower().strip() for l in label_str]

    valid = [(p, l) for p, l in zip(pred_str, label_str) if p and l]

    print("\n[Metric] sample predictions:")
    for pred_sample, label_sample in valid[:3]:
        print(f"  REF : {label_sample}")
        print(f"  PRED: {pred_sample}")
        print()

    if not valid:
        print("⚠️  WARNING: all predictions are empty! Check model generation config.")
        return {"wer": 1.0, "cer": 1.0, "n_valid": 0}

    preds_final  = [p for p, _ in valid]
    labels_final = [l for _, l in valid]

    wer = wer_metric.compute(predictions=preds_final, references=labels_final)
    cer = cer_metric.compute(predictions=preds_final, references=labels_final)

    return {
        "wer":     round(wer, 4),
        "cer":     round(cer, 4),
        "n_valid": len(valid),
    }


print("✅ Metrics loaded (WER + CER)")

2026-05-17 19:41:30.975003: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779046891.000985    2429 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779046891.009741    2429 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779046891.033444    2429 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779046891.033473    2429 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779046891.033476    2429 computation_placer.cc:177] computation placer alr

✅ Metrics loaded (WER + CER)


## 4. Pilot Training Run (~15 min)

Before committing to 3–5 hours of full training, run 100 steps on 600 samples. This catches configuration errors, loss divergence, WER metric bugs, and VRAM OOM issues.

**Expected pilot WER: 60–90%** (only 600 samples, 100 steps — purely a sanity check).

In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4.1 — Build pilot training arguments
# ─────────────────────────────────────────────────────────────────────────────

from transformers import Seq2SeqTrainingArguments

os.makedirs(cfg.pilot_output_dir, exist_ok=True)

pilot_args = Seq2SeqTrainingArguments(
    output_dir=cfg.pilot_output_dir,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    max_steps=cfg.pilot_max_steps,

    warmup_steps=10,
    logging_steps=10,
    eval_steps=50,
    save_steps=50,

    learning_rate=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    lr_scheduler_type=cfg.lr_scheduler_type,

    fp16=True,

    gradient_checkpointing=True,

    eval_strategy="steps",
    save_strategy="steps",

    # IMPORTANT
    predict_with_generate=False,

    dataloader_num_workers=2,

    load_best_model_at_end=False,
    report_to="tensorboard",
    push_to_hub=False,

    remove_unused_columns=False,
)

print("✅ Pilot training args configured")

✅ Pilot training args configured


In [19]:
model.config.use_cache = False

In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4.2 — Run pilot training
# ─────────────────────────────────────────────────────────────────────────────

from transformers import Seq2SeqTrainer

pilot_train = ds_train_proc.select(range(cfg.pilot_samples))
pilot_val   = ds_val_proc.select(range(min(100, len(ds_val_proc))))

pilot_trainer = Seq2SeqTrainer(
    model=model,
    args=pilot_args,
    data_collator=data_collator,
    train_dataset=pilot_train,
    eval_dataset=pilot_val,
    tokenizer=processor.feature_extractor,
)

print("🚀 Starting pilot training (100 steps, ~15 min)...")
pilot_result = pilot_trainer.train()
print("✅ Pilot complete")
print(pilot_result.metrics)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


🚀 Starting pilot training (100 steps, ~15 min)...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss
50,1.478900,1.537092
100,1.280600,1.365772


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch

✅ Pilot complete
{'train_runtime': 221.9302, 'train_samples_per_second': 7.209, 'train_steps_per_second': 0.451, 'total_flos': 3.9390216192e+16, 'train_loss': 1.5480523490905762, 'epoch': 2.6666666666666665}


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4.3 — Evaluate pilot on val set
# ─────────────────────────────────────────────────────────────────────────────

pilot_metrics = pilot_trainer.evaluate(pilot_val)
print("Pilot validation metrics:")
for k, v in pilot_metrics.items():
    print(f"  {k}: {v}")

assert pilot_metrics.get("eval_wer", 1.0) > 0.0, (
    "⚠️  WER=0.0 on pilot — likely a metric bug. Check compute_metrics logs above."
)
assert pilot_metrics.get("eval_loss", 0) < 10.0, (
    "⚠️  Loss is very high — check that preprocessing completed correctly."
)
print("\n✅ Pilot sanity checks passed. Proceeding to full training.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Pilot validation metrics:
  eval_loss: 1.3657715320587158
  eval_runtime: 8.4454
  eval_samples_per_second: 11.841
  eval_steps_per_second: 5.92
  epoch: 2.6666666666666665

✅ Pilot sanity checks passed. Proceeding to full training.


In [22]:
from transformers import Seq2SeqTrainer
import inspect

print(inspect.signature(Seq2SeqTrainer.__init__))

(self, model: Union[ForwardRef('PreTrainedModel'), torch.nn.modules.module.Module] = None, args: 'TrainingArguments' = None, data_collator: Optional[ForwardRef('DataCollator')] = None, train_dataset: Optional[torch.utils.data.dataset.Dataset] = None, eval_dataset: Union[torch.utils.data.dataset.Dataset, Dict[str, torch.utils.data.dataset.Dataset], NoneType] = None, tokenizer: Optional[ForwardRef('PreTrainedTokenizerBase')] = None, model_init: Optional[Callable[[], ForwardRef('PreTrainedModel')]] = None, compute_metrics: Optional[Callable[[ForwardRef('EvalPrediction')], Dict]] = None, callbacks: Optional[List[ForwardRef('TrainerCallback')]] = None, optimizers: Tuple[torch.optim.optimizer.Optimizer, torch.optim.lr_scheduler.LambdaLR] = (None, None), preprocess_logits_for_metrics: Optional[Callable[[torch.Tensor, torch.Tensor], torch.Tensor]] = None)


## 5. Full Training

**Runtime estimate on Kaggle T4:**
- ~0.3 it/s with batch_size=8, grad_accum=4  
- 4000 steps ÷ 0.3 ≈ **~3.7 hours**  
- Kaggle free sessions last ~9 hours — enough for one full run

### Checkpoint persistence on Kaggle
Unlike Colab, Kaggle **does not have Google Drive**. However:
- Checkpoints in `/kaggle/working/` are saved as **notebook output** and persist after the session ends.
- To resume in a new session: on the previous run's output page, click **"Add to project"** → this mounts it as a dataset input at `/kaggle/input/<output-name>/`.
- Then update `cfg.output_dir` to point there, or copy the checkpoint folder to `/kaggle/working/` at the start of the session.

The auto-resume cell (5.2) will find any checkpoint already present in `cfg.output_dir`.

In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5.1 — Full training arguments
# ─────────────────────────────────────────────────────────────────────────────

from transformers import Seq2SeqTrainingArguments

os.makedirs(cfg.output_dir,  exist_ok=True)
os.makedirs(cfg.logging_dir, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=cfg.output_dir,

    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,

    max_steps=cfg.max_steps,
    warmup_steps=cfg.warmup_steps,
    logging_steps=cfg.logging_steps,
    eval_steps=cfg.eval_steps,
    save_steps=cfg.save_steps,

    learning_rate=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    lr_scheduler_type=cfg.lr_scheduler_type,

    fp16=cfg.fp16,

    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=cfg.save_total_limit,
    load_best_model_at_end=cfg.load_best_model_at_end,
    metric_for_best_model=cfg.metric_for_best_model,
    greater_is_better=cfg.greater_is_better,

    predict_with_generate=True,
    generation_max_length=225,

    logging_dir=cfg.logging_dir,
    report_to="tensorboard",
    dataloader_num_workers=2,

    push_to_hub=False,  # we push manually in Section 8
)

print(f"✅ Training args configured")
print(f"   Output dir : {cfg.output_dir}")
print(f"   Max steps  : {cfg.max_steps}")
print(f"   Save every : {cfg.save_steps} steps")
print(f"   Eval every : {cfg.eval_steps} steps")

✅ Training args configured
   Output dir : /kaggle/working/whisper_torgo/checkpoints
   Max steps  : 2000
   Save every : 200 steps
   Eval every : 200 steps


In [24]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5.2 — HuggingFace Hub Checkpoint Callback
#
# Every `cfg.save_steps` steps the Trainer saves a local checkpoint.
# This callback ALSO pushes that checkpoint to your HF Hub repo so that
# even if Kaggle disconnects mid-run, all your progress is safe in the cloud.
#
# The checkpoint is pushed as a branch named  checkpoint-<step>  on the repo.
# To resume from a specific step later, just set:
#   RESUME_FROM_HF_STEP = <step>  in the resume cell below.
#
# Requirements: cfg.hub_model_id must be set, HF token must have WRITE access.
# ─────────────────────────────────────────────────────────────────────────────

import shutil
from transformers import TrainerCallback
from huggingface_hub import HfApi, create_repo

hf_api = HfApi()

# Create the repo now (no-op if it already exists)
create_repo(cfg.hub_model_id, repo_type="model", exist_ok=True)
print(f"✅ HF repo ready: https://huggingface.co/{cfg.hub_model_id}")


class HFCheckpointCallback(TrainerCallback):
    """
    After each checkpoint save, push the checkpoint folder to HF Hub as a
    branch named  checkpoint-<step>.  This makes every checkpoint cloud-safe
    and resumable from any machine — not just the current Kaggle session.
    """

    def __init__(self, hub_model_id: str, processor, push_every_n_steps: int = None,
                 keep_last_n: int = 3):
        self.hub_model_id = hub_model_id
        self.processor = processor
        # Default: push at the same frequency as save_steps
        self.push_every_n_steps = push_every_n_steps
        # Mirrors local save_total_limit — delete old branches beyond this window
        self.keep_last_n = keep_last_n

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        if self.push_every_n_steps and (step % self.push_every_n_steps != 0):
            return  # skip this save if not on the push schedule

        ckpt_dir = os.path.join(args.output_dir, f"checkpoint-{step}")
        if not os.path.isdir(ckpt_dir):
            print(f"⚠️  Checkpoint dir not found: {ckpt_dir}. Skipping HF push.")
            return

        branch = f"checkpoint-{step}"
        print(f"\n☁️  Pushing checkpoint-{step} to HF Hub (branch: {branch})...")

        try:
            # Save the processor alongside the model weights in the checkpoint folder
            # so the branch is fully self-contained for resuming
            self.processor.save_pretrained(ckpt_dir)

            # Create the branch on HF Hub (no-op if it already exists)
            try:
                hf_api.create_branch(
                    repo_id=self.hub_model_id,
                    branch=branch,
                    repo_type="model",
                    exist_ok=True,
                )
            except Exception:
                pass  # branch may already exist from a previous push of same step

            # Upload the entire checkpoint folder to this branch
            hf_api.upload_folder(
                folder_path=ckpt_dir,
                repo_id=self.hub_model_id,
                repo_type="model",
                revision=branch,
                commit_message=f"Auto-checkpoint at step {step}",
            )
            print(f"   ✅ Checkpoint-{step} saved to "
                  f"https://huggingface.co/{self.hub_model_id}/tree/{branch}")

            # Also update 'main' branch trainer_state.json so we can inspect
            # which step is latest without downloading weights
            state_path = os.path.join(ckpt_dir, "trainer_state.json")
            if os.path.exists(state_path):
                hf_api.upload_file(
                    path_or_fileobj=state_path,
                    path_in_repo="trainer_state.json",
                    repo_id=self.hub_model_id,
                    repo_type="model",
                    revision="main",
                    commit_message=f"Update trainer_state.json (step {step})",
                )

        except Exception as e:
            print(f"   ⚠️  HF push failed at step {step}: {e}")
            print("      Training continues — checkpoint is still local.")

        # ── Delete old branches to mirror local save_total_limit ──────────────
        # keep_last_n=3 means we retain the 3 most recent checkpoint branches.
        # Branches are named checkpoint-<step> at multiples of save_steps.
        old_step = step - (self.keep_last_n * args.save_steps)
        if old_step > 0:
            old_branch = f"checkpoint-{old_step}"
            try:
                hf_api.delete_branch(
                    repo_id=self.hub_model_id,
                    branch=old_branch,
                    repo_type="model",
                )
                print(f"   🗑️  Deleted old branch {old_branch} (keeping last {self.keep_last_n})")
            except Exception:
                pass  # branch doesn't exist or already deleted — ignore


hf_checkpoint_callback = HFCheckpointCallback(
    hub_model_id=cfg.hub_model_id,
    processor=processor,
    push_every_n_steps=cfg.save_steps,  # same cadence as local saves
)

print(f"✅ HFCheckpointCallback ready")
print(f"   Will push every {cfg.save_steps} steps → {cfg.hub_model_id}")


✅ HF repo ready: https://huggingface.co/ruch9265/distil-whisper-torgo
✅ HFCheckpointCallback ready
   Will push every 200 steps → ruch9265/distil-whisper-torgo


In [25]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5.3 — Auto-resume: local first, then HF Hub fallback
#
# Priority order:
#   1. Latest local checkpoint in cfg.output_dir  (same session)
#   2. Specific step from HF Hub  (set RESUME_FROM_HF_STEP)
#   3. Latest checkpoint on HF Hub 'main' branch trainer_state.json
#   4. Fresh start from pretrained weights
#
# TO RESUME AFTER A DISCONNECTION:
#   Option A — Automatic: just re-run this cell. It reads trainer_state.json
#              from HF Hub to find the latest step and downloads that branch.
#   Option B — Manual: set RESUME_FROM_HF_STEP = <step> to force a specific one.
# ─────────────────────────────────────────────────────────────────────────────

import glob, json, shutil
from huggingface_hub import hf_hub_download, snapshot_download, list_repo_refs
from huggingface_hub.utils import RepositoryNotFoundError, EntryNotFoundError

# ── Config ────────────────────────────────────────────────────────────────────
# Set to an integer (e.g. 1000) to force-resume from a specific HF checkpoint.
# Leave as None for automatic detection.
RESUME_FROM_HF_STEP = None
# ─────────────────────────────────────────────────────────────────────────────


def find_latest_local_checkpoint(output_dir: str):
    pattern = os.path.join(output_dir, "checkpoint-*")
    checkpoints = glob.glob(pattern)
    if not checkpoints:
        return None
    checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    return checkpoints[-1]


def find_latest_hf_step(hub_model_id: str) -> int | None:
    """Read trainer_state.json from HF Hub 'main' to get the last saved step."""
    try:
        state_file = hf_hub_download(
            repo_id=hub_model_id,
            filename="trainer_state.json",
            repo_type="model",
        )
        with open(state_file) as f:
            state = json.load(f)
        step = state.get("global_step", None)
        print(f"   HF Hub trainer_state.json → global_step = {step}")
        return step
    except (RepositoryNotFoundError, EntryNotFoundError, Exception) as e:
        print(f"   Could not read trainer_state.json from HF Hub: {e}")
        return None


def download_hf_checkpoint(hub_model_id: str, step: int, local_dir: str) -> str:
    """Download checkpoint-<step> branch from HF Hub to local_dir."""
    branch = f"checkpoint-{step}"
    dest   = os.path.join(local_dir, f"checkpoint-{step}")
    if os.path.isdir(dest):
        print(f"   Checkpoint-{step} already present locally at {dest}")
        return dest

    print(f"   Downloading checkpoint-{step} from HF Hub (branch: {branch})...")
    os.makedirs(dest, exist_ok=True)
    snapshot_download(
        repo_id=hub_model_id,
        revision=branch,
        repo_type="model",
        local_dir=dest,
        ignore_patterns=["*.msgpack", "flax_model*"],  # skip non-PyTorch weights
    )
    print(f"   ✅ Downloaded to {dest}")
    return dest


# ── Resume detection ──────────────────────────────────────────────────────────
resume_from = None

# 1. Local checkpoint?
local_ckpt = find_latest_local_checkpoint(cfg.output_dir)
if local_ckpt:
    print(f"♻️  Found local checkpoint: {local_ckpt}")
    resume_from = local_ckpt

else:
    print("No local checkpoint found. Checking HF Hub...")

    # Determine which step to pull
    hf_step = RESUME_FROM_HF_STEP
    if hf_step is None:
        hf_step = find_latest_hf_step(cfg.hub_model_id)

    if hf_step:
        try:
            resume_from = download_hf_checkpoint(
                hub_model_id=cfg.hub_model_id,
                step=hf_step,
                local_dir=cfg.output_dir,
            )
            print(f"♻️  Resuming from HF Hub checkpoint-{hf_step}")
        except Exception as e:
            print(f"⚠️  Could not download HF checkpoint: {e}")
            resume_from = None
    else:
        print("🆕 No HF Hub checkpoint found — starting fresh from pretrained weights")

if resume_from:
    print(f"\n✅ Will resume training from: {resume_from}")
else:
    print("\n✅ Starting fresh — no checkpoint to resume from")


No local checkpoint found. Checking HF Hub...
   Could not read trainer_state.json from HF Hub: 404 Client Error. (Request ID: Root=1-6a0a1ad8-4fcefdb7275812b93a8267ac;be5a72f3-b08f-4a8a-827d-217c93ff5238)

Entry Not Found for url: https://huggingface.co/ruch9265/distil-whisper-torgo/resolve/main/trainer_state.json.
🆕 No HF Hub checkpoint found — starting fresh from pretrained weights

✅ Starting fresh — no checkpoint to resume from


In [26]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5.3 — Instantiate Seq2SeqTrainer and start full training
#
# VRAM budget on T4 (15-16 GB):
#   Model weights (FP32)         : ~0.8 GB
#   AMP fp16 copies              : ~0.8 GB
#   Gradient buffers (FP32)      : ~0.8 GB
#   Adam optimiser states        : ~1.6 GB
#   Activations (batch_size=8)   : ~4–6 GB
#   Feature extractor output     : ~0.5 GB
#   Safety margin                : ~3 GB
#   TOTAL                        : ~10–12 GB  ✅ under 15 GB
#
# If you hit OOM: reduce per_device_train_batch_size to 4,
# increase gradient_accumulation_steps to 8 (effective BS stays 32).
# ─────────────────────────────────────────────────────────────────────────────

from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_train_proc,
    eval_dataset=ds_val_proc,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,  # not tokenizer — deprecated
    callbacks=[hf_checkpoint_callback],      # ← auto-push checkpoints to HF Hub
)

print("🚀 Starting full training...")
print(f"   Train samples : {len(ds_train_proc):,}")
print(f"   Val samples   : {len(ds_val_proc):,}")
print(f"   Max steps     : {cfg.max_steps}")
print(f"   Effective BS  : {cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps}")
print(f"   Checkpoints → : {cfg.output_dir}")
print()

train_result = trainer.train(resume_from_checkpoint=resume_from)

print("\n✅ Training complete!")
print(train_result.metrics)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


🚀 Starting full training...
   Train samples : 13,239
   Val samples   : 1,656
   Max steps     : 2000
   Effective BS  : 16
   Checkpoints → : /kaggle/working/whisper_torgo/checkpoints



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Wer,Cer,N Valid
200,0.532000,0.514257,0.341200,0.197000,1656
400,0.453200,0.427411,0.255400,0.138700,1656
600,0.418100,0.391413,0.241600,0.161600,1656
800,0.323400,0.367177,0.229400,0.154700,1656
1000,0.314800,0.349728,0.220500,0.151100,1656
1200,0.285100,0.337504,0.206100,0.122300,1656
1400,0.280900,0.327955,0.205000,0.122400,1656
1600,0.255500,0.324005,0.202200,0.110100,1655
1800,0.251800,0.320320,0.202000,0.121000,1655
2000,0.260800,0.319047,0.201000,0.121000,1655


You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, 50259], [2, 50359], [3, 50363]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



[Metric] sample predictions:
  REF : quart
  PRED: court

  REF : lair
  PRED: layer

  REF : brought
  PRED: brought



... (97 outputs left out)

scheduler.pt:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

   ✅ Checkpoint-2000 saved to https://huggingface.co/ruch9265/distil-whisper-torgo/tree/checkpoint-2000
   🗑️  Deleted old branch checkpoint-1400 (keeping last 3)


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].



✅ Training complete!
{'train_runtime': 7348.337, 'train_samples_per_second': 8.709, 'train_steps_per_second': 0.272, 'total_flos': 1.57472236781568e+18, 'train_loss': 0.37718193006515505, 'epoch': 4.830917874396135}


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5.4 — Save final model
#
# KAGGLE: saved to /kaggle/working/whisper_torgo/final_model/
# This will appear in your notebook's Output tab after the session.
# ─────────────────────────────────────────────────────────────────────────────

final_model_dir = os.path.join(WORKING_BASE, "final_model")
os.makedirs(final_model_dir, exist_ok=True)

trainer.save_model(final_model_dir)
processor.save_pretrained(final_model_dir)

trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

print(f"✅ Final model saved to {final_model_dir}")

## 6. Evaluation on Test Set

In [31]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6.1 — Full test set evaluation
# ─────────────────────────────────────────────────────────────────────────────

print("Evaluating on held-out test set...")
test_metrics = trainer.evaluate(ds_test_proc, metric_key_prefix="test")

print("\n" + "=" * 50)
print("TEST RESULTS")
print("=" * 50)
for k, v in test_metrics.items():
    print(f"  {k}: {v}")

trainer.log_metrics("test", test_metrics)
trainer.save_metrics("test", test_metrics)

Evaluating on held-out test set...


Exception ignored in: Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b03c225dd00>
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 270, in _feed
    send_bytes(obj)
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 200, in send_bytes
    self._send_bytes(m[offset:offset + size])
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 427, in _send_bytes
    self._send(header + buf)
Exception ignored in:       File "/usr/lib/python3.12/multiprocessing/queues.py", line 270, in _feed
    send_bytes(obj)
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b03c225dd00>  File "/usr/lib/python3.12/multiprocessing/connection.py", line 384, in _send
    n = write(self._handle, buf)
        ^^^^^^^^^^^^^^^^^^^^^^^^
self._shutdown_workers()  File


[Metric] sample predictions:
  REF : my sister made the flowered curtains
  PRED: my sister made the flowered curtains

  REF : pile
  PRED: par

  REF : pad
  PRED: pat


TEST RESULTS
  test_loss: 0.32547739148139954
  test_wer: 0.2032
  test_cer: 0.1067
  test_n_valid: 1655
  test_runtime: 198.3341
  test_samples_per_second: 8.345
  test_steps_per_second: 0.524
  epoch: 4.830917874396135
***** test metrics *****
  epoch                   =     4.8309
  test_cer                =     0.1067
  test_loss               =     0.3255
  test_n_valid            =       1655
  test_runtime            = 0:03:18.33
  test_samples_per_second =      8.345
  test_steps_per_second   =      0.524
  test_wer                =     0.2032


In [33]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6.2 — Breakdown by speech_status (healthy vs dysarthric)
# ─────────────────────────────────────────────────────────────────────────────

import torch
from transformers import pipeline as hf_pipeline

asr_pipe = hf_pipeline(
    "automatic-speech-recognition",
    model=trainer.model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch.float16,
    device=0,
    generate_kwargs={"max_new_tokens": 225, "language": cfg.language, "task": cfg.task},
)


def evaluate_subset(ds_raw, label: str, n_samples: int = 200):
    subset = ds_raw.select(range(min(n_samples, len(ds_raw))))
    preds, refs = [], []
    for sample in subset:
        audio_array = sample["audio"]["array"]
        result = asr_pipe({"raw": audio_array, "sampling_rate": cfg.sampling_rate})
        pred = result["text"].lower().strip()
        ref  = sample["transcription"].lower().strip()
        if pred and ref:
            preds.append(pred)
            refs.append(ref)
    wer = wer_metric.compute(predictions=preds, references=refs)
    cer = cer_metric.compute(predictions=preds, references=refs)
    print(f"[{label}] WER={wer:.4f}  CER={cer:.4f}  (n={len(preds)})")
    return wer, cer


test_healthy    = ds_test.filter(lambda x: x["speech_status"] == "healthy")
test_dysarthric = ds_test.filter(lambda x: x["speech_status"] == "dysarthria")

print(f"Test healthy    : {len(test_healthy)} samples")
print(f"Test dysarthric : {len(test_dysarthric)} samples")
print()

evaluate_subset(test_healthy,    "Healthy   ")
evaluate_subset(test_dysarthric, "Dysarthric")

Test healthy    : 1120 samples
Test dysarthric : 535 samples

[Healthy   ] WER=0.1241  CER=0.0623  (n=200)
[Dysarthric] WER=0.3503  CER=0.2039  (n=200)


(0.3502722323049002, 0.20387994143484628)

## 7. Inference Demo

In [34]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7.1 — Transcribe a random test sample
# ─────────────────────────────────────────────────────────────────────────────

import random
import numpy as np
from IPython.display import Audio as IPyAudio, display

idx    = random.randint(0, len(test_dysarthric) - 1)
sample = test_dysarthric[idx]

audio_array  = sample["audio"]["array"]
sample_rate  = sample["audio"]["sampling_rate"]
ground_truth = sample["transcription"]

print(f"Sample info:")
print(f"  Status  : {sample['speech_status']}")
print(f"  Gender  : {sample['gender']}")
print(f"  Duration: {len(audio_array)/sample_rate:.2f} s")
print(f"  REF     : {ground_truth}")

# Play inline — works in both Kaggle and Colab notebook environments
display(IPyAudio(audio_array, rate=sample_rate))

result = asr_pipe({"raw": audio_array, "sampling_rate": sample_rate})
print(f"  PRED    : {result['text']}")

Sample info:
  Status  : dysarthria
  Gender  : female
  Duration: 2.37 s
  REF     : chair


  PRED    : chair


In [41]:
import os

base_path = "/kaggle/input/datasets/rshar21/test-audio-wav"

print(os.listdir(base_path))

['temp_audio.wav']


In [42]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7.2 — Transcribe a local audio file
#
# KAGGLE NOTE: Kaggle kernels have no interactive file upload dialog
# (google.colab.files is unavailable). Instead:
#   Option A — attach an audio file as a Kaggle dataset and read from
#              /kaggle/input/<dataset-name>/your_file.wav
#   Option B — paste a public URL and download it with wget/requests
#
# Replace AUDIO_PATH below with the actual path to your file.
# ─────────────────────────────────────────────────────────────────────────────

import soundfile as sf

# ── Option A: file from attached dataset ──────────────────────────────────────
# AUDIO_PATH = "/kaggle/input/my-audio-dataset/speech_sample.wav"

# ── Option B: download from URL ───────────────────────────────────────────────
# !wget -q -O /kaggle/working/sample.wav "https://example.com/sample.wav"
# AUDIO_PATH = "/kaggle/working/sample.wav"

AUDIO_PATH = "/kaggle/input/datasets/rshar21/test-audio-wav/temp_audio.wav"  # ← set this before running

if AUDIO_PATH and os.path.exists(AUDIO_PATH):
    waveform, sr = sf.read(AUDIO_PATH)

    # Convert stereo → mono if needed
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)

    print(f"File: {AUDIO_PATH}")
    print(f"  Sample rate: {sr} Hz")
    print(f"  Duration   : {len(waveform)/sr:.2f} s")

    result = asr_pipe({"raw": waveform, "sampling_rate": sr})
    print(f"  Transcription: {result['text']}")
else:
    print("ℹ️  Set AUDIO_PATH to a valid file path to transcribe your own audio.")
    print("   See comments above for how to attach files on Kaggle.")

File: /kaggle/input/datasets/rshar21/test-audio-wav/temp_audio.wav
  Sample rate: 16000 Hz
  Duration   : 17.36 s
  Transcription: He further proposed giants of an unexpected sum of "experimental" offers


In [44]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7.3 — Direct model inference (no pipeline)
# Useful for debugging or when you need more control over generation.
# ─────────────────────────────────────────────────────────────────────────────

import torch
import numpy as np

def transcribe_direct(audio_array: np.ndarray, sr: int = 16_000) -> str:
    inputs = processor(audio_array, sampling_rate=sr, return_tensors="pt")
    input_features = inputs.input_features.to("cuda")

    with torch.no_grad():
        predicted_ids = trainer.model.generate(
            input_features,
            max_new_tokens=225,
            forced_decoder_ids=None,
        )

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
    return transcription[0]


sample = test_dysarthric[0]
pred = transcribe_direct(sample["audio"]["array"], sr=cfg.sampling_rate)
print(f"REF : {sample['transcription']}")
print(f"PRED: {pred}")

REF : pile
PRED: par


## 8. Push to Hugging Face Hub

In [45]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8.1 — Upload model + processor to HF Hub
#
# Make sure cfg.hub_model_id is set correctly in Section 1.
# Your HF token must have WRITE access (set via Kaggle Secret HF_TOKEN).
#
# NOTE: input() works in Kaggle notebooks — the confirmation prompt is fine.
# ─────────────────────────────────────────────────────────────────────────────

from huggingface_hub import HfApi

print(f"Pushing to: https://huggingface.co/{cfg.hub_model_id}")
confirm = input("Confirm push? (yes/no): ")

if confirm.strip().lower() == "yes":
    model_card = f"""---
language: en
license: apache-2.0
base_model: {cfg.model_name}
datasets:
  - {cfg.dataset_name}
tags:
  - whisper
  - asr
  - dysarthric-speech
  - speech-recognition
metrics:
  - wer
  - cer
---

# Distil-Whisper Fine-tuned on TORGO Dysarthric Speech

Fine-tuned from [{cfg.model_name}](https://huggingface.co/{cfg.model_name}) on the
[TORGO database](https://huggingface.co/datasets/{cfg.dataset_name}) for improved
recognition of dysarthric (motor-impaired) speech.

## Training Details
- Dataset: {cfg.dataset_name} ({len(ds_train_proc):,} train / {len(ds_val_proc):,} val / {len(ds_test_proc):,} test)
- Effective batch size: {cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps}
- Learning rate: {cfg.learning_rate}
- Steps: {cfg.max_steps}
- FP16 mixed precision
- Full encoder + decoder fine-tuning

## Usage
```python
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="{cfg.hub_model_id}")
result = asr("path/to/audio.wav")
print(result["text"])
```
"""

    with open(os.path.join(final_model_dir, "README.md"), "w") as f:
        f.write(model_card)

    trainer.model.push_to_hub(
        cfg.hub_model_id,
        commit_message="Fine-tuned on TORGO dysarthric speech",
    )

    processor.push_to_hub(cfg.hub_model_id)

    api = HfApi()
    api.upload_file(
        path_or_fileobj=os.path.join(final_model_dir, "README.md"),
        path_in_repo="README.md",
        repo_id=cfg.hub_model_id,
        repo_type="model",
    )

    print(f"✅ Model pushed to https://huggingface.co/{cfg.hub_model_id}")
else:
    print("Push cancelled.")

Pushing to: https://huggingface.co/ruch9265/distil-whisper-torgo


Confirm push? (yes/no):  yes


README.md: 0.00B [00:00, ?B/s]

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}


✅ Model pushed to https://huggingface.co/ruch9265/distil-whisper-torgo


## 9. Troubleshooting Guide

### 9.1 — FP16 Gradient Unscaling Crash

**Error:** `ValueError: Attempting to unscale FP16 gradients`

**Cause:** Model loaded with `torch_dtype=torch.float16` + fp16 training. GradScaler expects FP32 master weights.

**Fix:** Remove `torch_dtype=torch.float16` from `from_pretrained()`.

### 9.2 — WER is 0.0

**Cause A:** Predictions are empty strings → jiwer returns 0 for empty pairs.  
**Cause B:** Metric computed on token-ids instead of decoded strings.  
**Cause C:** Label ids not un-masked before decode (−100 tokens passed to tokenizer).

**Fix:** The `compute_metrics` function in Cell 3.2 handles all three. Check the printed sample predictions.

### 9.3 — Empty Predictions / Generation Collapse

| Cause | Fix |
|-------|-----|
| `forced_decoder_ids` still set | `model.generation_config.forced_decoder_ids = None` |
| `suppress_tokens` blocking outputs | `model.generation_config.suppress_tokens = []` |
| `max_length` conflict with prompt | Use `max_new_tokens=225` instead |
| Language/task tokens not set | Set `language` and `task` in `generation_config` |

### 9.4 — OOM (Out of Memory)

Reduce batch size without changing effective batch size:
```python
cfg.per_device_train_batch_size = 4
cfg.gradient_accumulation_steps = 8  # effective BS stays 32
```

### 9.5 — Very Slow Training (<0.1 it/s)

1. Ensure preprocessing (Cell 2.7) completed with `num_proc=2`
2. Check `dataloader_num_workers=2` is set in training args (it is)
3. Confirm GPU is enabled (Cell 0.4)

### 9.6 — Session Expired Mid-Training (Kaggle-specific)

Kaggle free tier sessions last ~9 hours. If your session expires:

1. Start a new session and re-run Sections 0–3
2. **Skip Section 4** (pilot)
3. Go to **Cell 5.2** — it will find the latest checkpoint in `/kaggle/working/`
4. If the checkpoint was saved in a *previous* session's output, attach that output as a dataset input and copy it:
   ```python
   !cp -r /kaggle/input/<output-name>/whisper_torgo/checkpoints /kaggle/working/whisper_torgo/
   ```
5. Then run Cell 5.2 onward — training resumes automatically

### 9.7 — Quick VRAM Optimisation Reference

| Technique | VRAM saving | Applied? |
|-----------|-------------|----------|
| FP16 AMP | ~50% activations | ✅ `fp16=True` |
| Gradient accumulation | Reduces peak batch VRAM | ✅ `grad_accum=4` |
| Gradient checkpointing | ~30-40% at speed cost | ❌ (incompatible with fp16+full encoder) |
| Freeze encoder | ~40% fewer gradient buffers | ❌ (hurts dysarthric adaptation) |
| LoRA / PEFT | 10-100× fewer trainable params | Optional (see below) |

#### LoRA alternative (if you still hit OOM)

```python
# pip install peft
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# trainable params: ~2.4M vs 756M total — 99.7% reduction
```


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9.1 — Verify model weight dtype (should be float32)
# ─────────────────────────────────────────────────────────────────────────────

first_param = next(model.parameters())
print(f"Model weight dtype: {first_param.dtype}")
assert first_param.dtype == torch.float32, (
    "❌ Model weights are not FP32. Remove torch_dtype=torch.float16 from from_pretrained()."
)
print("✅ Model weights are FP32 — safe for fp16 AMP training")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9.2 — Manually check what the model generates on a known sample
# ─────────────────────────────────────────────────────────────────────────────

sample = ds_test[0]
result = asr_pipe({"raw": sample["audio"]["array"], "sampling_rate": cfg.sampling_rate})
pred   = result["text"]
ref    = sample["transcription"]

print(f"REF : '{ref}'")
print(f"PRED: '{pred}'")

if not pred.strip():
    print("\n⚠️  Empty prediction! See troubleshooting 9.3")
else:
    from jiwer import wer as compute_wer
    print(f"\nSingle-sample WER: {compute_wer(ref.lower(), pred.lower()):.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9.3 — Inspect generation config
# ─────────────────────────────────────────────────────────────────────────────

gc = trainer.model.generation_config
print("Generation config:")
print(f"  forced_decoder_ids : {gc.forced_decoder_ids}")
print(f"  suppress_tokens    : {gc.suppress_tokens}")
print(f"  language           : {gc.language}")
print(f"  task               : {gc.task}")
print(f"  max_new_tokens     : {getattr(gc, 'max_new_tokens', 'not set')}")

# Apply fixes if needed
gc.forced_decoder_ids = None
gc.suppress_tokens = []
print("\n✅ forced_decoder_ids and suppress_tokens reset")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9.4 — Check current VRAM usage
# ─────────────────────────────────────────────────────────────────────────────

import torch

allocated = torch.cuda.memory_allocated(0) / 1e9
reserved  = torch.cuda.memory_reserved(0)  / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"VRAM allocated : {allocated:.2f} GB")
print(f"VRAM reserved  : {reserved:.2f} GB")
print(f"VRAM total     : {total:.2f} GB")
print(f"VRAM free      : {total - reserved:.2f} GB")

if reserved > 12.0:
    print("\n⚠️  VRAM usage is high. Consider:")
    print("   cfg.per_device_train_batch_size = 4")
    print("   cfg.gradient_accumulation_steps = 8")
    print("   (effective batch size stays at 32)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9.5 — Benchmark a single training step
# ─────────────────────────────────────────────────────────────────────────────

import time
import torch
from torch.utils.data import DataLoader

bench_loader = DataLoader(
    ds_train_proc.select(range(16)),
    batch_size=8,
    collate_fn=data_collator,
    num_workers=0,
)
batch = next(iter(bench_loader))
batch = {k: v.to("cuda") for k, v in batch.items()}

model.train()
t0 = time.time()
with torch.cuda.amp.autocast():
    out = model(**batch)
loss = out.loss
loss.backward()
t1 = time.time()

print(f"Single step (forward + backward): {t1-t0:.2f} s")
print(f"Estimated throughput: {1/(t1-t0):.2f} steps/s")
print(f"Loss: {loss.item():.4f}")